In [1]:
pip install --upgrade google-api-python-client google-auth-httplib2 google-auth-oauthlib

Note: you may need to restart the kernel to use updated packages.Requirement already satisfied: google-api-python-client in c:\users\saiva\anaconda3\lib\site-packages (2.193.0)



In [ ]:
import os.path
import base64
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build

In [18]:
SCOPES = ['https://www.googleapis.com/auth/gmail.readonly']

In [19]:
creds = None

if os.path.exists('token.json'):
    print("Found an existing badge (token.json)!")
    creds = Credentials.from_authorized_user_file('token.json', SCOPES)

if not creds or not creds.valid:
    print("No valid badge found or it expired. Updating...")
    
    if creds and creds.expired and creds.refresh_token:
        creds.refresh(Request())
    else:
        flow = InstalledAppFlow.from_client_secrets_file('credentials.json', SCOPES)
        creds = flow.run_local_server(port=0)
    
    with open('token.json', 'w') as token:
        token.write(creds.to_json())
        
    print("Success! VIP badge saved to your computer as 'token.json'.")

print("Phase 1 Complete! The 'creds' variable now holds your active VIP badge.")

Found an existing badge (token.json)!
No valid badge found or it expired. Updating...
Success! VIP badge saved to your computer as 'token.json'.
Phase 1 Complete! The 'creds' variable now holds your active VIP badge.


In [2]:

service = build('gmail', 'v1', credentials=creds)

print("Translator is awake and connected to your Gmail Vault!")

NameError: name 'build' is not defined

In [21]:
def get_clean_text(payload):
    """Digs through the messy email package to find the plain text."""
    
    if 'data' in payload.get('body', {}):
        scrambled_data = payload['body']['data']
        return base64.urlsafe_b64decode(scrambled_data).decode('utf-8')
    
    if 'parts' in payload:
        for part in payload['parts']:
            if part['mimeType'] == 'text/plain': # Ignore HTML/Images
                scrambled_data = part['body']['data']
                return base64.urlsafe_b64decode(scrambled_data).decode('utf-8')
            
    return "Could not find plain text body."

print("Unscrambler tool is ready!")

Unscrambler tool is ready!


In [12]:

results = service.users().messages().list(userId='me', maxResults=1).execute()
messages = results.get('messages', [])

if not messages:
    print('Your inbox is completely empty.')
else:
    newest_email_id = messages[0]['id']

    msg_data = service.users().messages().get(userId='me', id=newest_email_id, format='full').execute()
    payload = msg_data['payload']

    headers = payload['headers']
    subject = "No Subject Found"
    for header in headers:
        if header['name'] == 'Subject':
            subject = header['value']
            break

    body_text = get_clean_text(payload)

    print("\n📬 NEWEST EMAIL DETAILS:")
    print(f"Subject: {subject}")
    print("-" * 40)
    print(f"Body Preview:\n{body_text[:500]}...") # Printing just the first 500 characters


📬 NEWEST EMAIL DETAILS:
Subject: Re: Pixel Compute Online test is scheduled on 06th April 2026 @ CDC Labs
----------------------------------------
Body Preview:
Reminder to all

*PPT Venue - CB305*
*Reporting Time - 09:00 AM*

*All shortlisted students are required to report to CB-305 by 09:00 AM for
the PPT session  *

*Formal Dress Mandatory*

On Fri, Apr 3, 2026 at 8:32 PM PAT VIT-AP <placement@vitap.ac.in> wrote:

> Dear Students,
>
> PFA of shortlisted students list and venue details
>
> *6th April 2026*
>
>    - *9:15 AM* – Pre-placement talk for all applied students. Venue:
>    CB305, Reporting time: 9.00AM
>
>
>    - *10:3...


In [13]:
import sqlite3

conn = sqlite3.connect('rag_database.db')
cursor = conn.cursor()

cursor.execute('''
    CREATE TABLE IF NOT EXISTS emails (
        id TEXT PRIMARY KEY,
        subject TEXT,
        body TEXT,
        timestamp INTEGER
    )
''')
conn.commit()

In [14]:
def chunk_text(text, chunk_size=100, overlap=20):
    """Chops long text into overlapping chunks so the AI doesn't lose memory."""
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunks.append(" ".join(words[i : i + chunk_size]))
    return chunks

print("✅ Imports and Tools loaded!")

✅ Imports and Tools loaded!


In [15]:
import time
import json
import sqlite3

SYNC_FILE = 'last_sync.json'
INITIAL_FETCH_LIMIT = 100

print(f"🚀 BOOTSTRAP: Fetching the initial {INITIAL_FETCH_LIMIT} emails...")

results = service.users().messages().list(userId='me', maxResults=INITIAL_FETCH_LIMIT).execute()
messages = results.get('messages', [])

if not messages:
    print("Your inbox is empty. Nothing to fetch.")
else:
    conn = sqlite3.connect('rag_database.db')
    cursor = conn.cursor()
    saved_count = 0

    for message in messages:
        try:
            msg_id = message['id']
            msg_data = service.users().messages().get(userId='me', id=msg_id, format='full').execute()
            payload = msg_data['payload']

            headers = payload.get('headers', [])
            subject = next((h['value'] for h in headers if h['name'] == 'Subject'), "No Subject")
            body_text = get_clean_text(payload)
            
            cursor.execute('''
                INSERT OR IGNORE INTO emails (id, subject, body, timestamp)
                VALUES (?, ?, ?, ?)
            ''', (msg_id, subject, body_text, int(time.time())))
            saved_count += 1
        except Exception as e:
            pass

    conn.commit()
    conn.close()

    with open(SYNC_FILE, 'w') as f:
        json.dump({'last_sync_time': int(time.time())}, f)

    print(f"✅ Backfill Complete! {saved_count} emails saved.")
    print("⏰ The sync clock has been started. You are ready for Delta Syncs.")

🚀 BOOTSTRAP: Fetching the initial 100 emails...
✅ Backfill Complete! 100 emails saved.
⏰ The sync clock has been started. You are ready for Delta Syncs.


In [1]:
import time
import json
import sqlite3
import os

SYNC_FILE = 'last_sync.json'

print("🔄 Waking up Delta Sync Worker...")

if not os.path.exists(SYNC_FILE):
    print("⚠️ Error: No sync clock found. Please run the Backfill cell first!")
else:
    with open(SYNC_FILE, 'r') as f:
        last_sync = json.load(f).get('last_sync_time')

    print(f"Looking for emails received after timestamp: {last_sync}")
    
    results = service.users().messages().list(userId='me', q=f"after:{last_sync}").execute()
    messages = results.get('messages', [])

    if not messages:
        print("✅ Inbox is completely up to date. Going back to sleep.")
    else:
        conn = sqlite3.connect('rag_database.db')
        cursor = conn.cursor()
        saved_count = 0
        
        # 4. Process and Save
        for message in messages:
            try:
                msg_id = message['id']
                msg_data = service.users().messages().get(userId='me', id=msg_id, format='full').execute()
                payload = msg_data['payload']

                headers = payload.get('headers', [])
                subject = next((h['value'] for h in headers if h['name'] == 'Subject'), "No Subject")
                body_text = get_clean_text(payload)
                
                cursor.execute('''
                    INSERT OR IGNORE INTO emails (id, subject, body, timestamp)
                    VALUES (?, ?, ?, ?)
                ''', (msg_id, subject, body_text, int(time.time())))
                saved_count += 1
            except Exception as e:
                pass 

        conn.commit()
        conn.close()

        print(f"✅ Delta Sync Complete! Added {saved_count} new emails.")

    # 5. Reset the clock for next time
    with open(SYNC_FILE, 'w') as f:
        json.dump({'last_sync_time': int(time.time())}, f)
    print("🕒 Clock reset.")

🔄 Waking up Delta Sync Worker...
Looking for emails received after timestamp: 1775834560


NameError: name 'service' is not defined

In [2]:
import sqlite3

conn = sqlite3.connect("rag_database.db")
cursor = conn.cursor()

# show all tables
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
print(cursor.fetchall())

[('emails',)]


In [4]:
cursor.execute("SELECT * FROM emails LIMIT 5;")
rows = cursor.fetchall()

for row in rows:
    print(row)

('19d5e1072890dd99', 'Re: Pixel Compute Online test is scheduled on 06th April 2026 @ CDC Labs', "Reminder to all\r\n\r\n*PPT Venue - CB305*\r\n*Reporting Time - 09:00 AM*\r\n\r\n*All shortlisted students are required to report to CB-305 by 09:00 AM for\r\nthe PPT session  *\r\n\r\n*Formal Dress Mandatory*\r\n\r\nOn Fri, Apr 3, 2026 at 8:32\u202fPM PAT VIT-AP <placement@vitap.ac.in> wrote:\r\n\r\n> Dear Students,\r\n>\r\n> PFA of shortlisted students list and venue details\r\n>\r\n> *6th April 2026*\r\n>\r\n>    - *9:15 AM* – Pre-placement talk for all applied students. Venue:\r\n>    CB305, Reporting time: 9.00AM\r\n>\r\n>\r\n>    - *10:30 AM* – Online assessment (Batch 1) - eligible if pre placement\r\n>    talk is attended\r\n>\r\n>\r\n>    - *2:00 PM* – Online assessment (Batch 2) - eligible only if pre\r\n>    placement talk is attended.\r\n>\r\n> The online assessment will have:\r\n>\r\n>    - DSA round (90 minutes)\r\n>\r\n>\r\n>    - MCQ round (30 minutes)\r\n>    - Batch 1 res